# 02F: Regime-Filtered Meta-Signal

Combines the **T+1 next-day signal (02D)** and the **T+15 medium-term signal (02E)**
into a single, regime-aware trading signal that suppresses false breakouts.

## Why combine T+1 and T+15?

| Problem | Solution |
|---|---|
| T+1 (02D) is noisy on single-headline spikes | T+15 (02E) sees through single-day noise |
| T+15 reacts slowly to sudden regime changes | T+1 (02D) captures immediate price response |
| High-VIX periods make T+1 unreliable | VIX-conditional weighting shifts trust to T+15 |
| Sentiment velocity ↓ but T+1 predicts ↑ | Velocity alignment filter discounts T+1 |

## Three filtering mechanisms
1. **Regime Filter**: If T+15 is negative, discount any positive T+1 signal  
2. **Sentiment Velocity Alignment**: Rising narrative momentum must align with T+1 direction  
3. **VIX-Conditional Weighting**: High VIX → trust T+15 more; low VIX → trust T+1 more  

## Output files
- `02F_meta_signal_predictions.csv` — combined daily signal with weights
- `02F_backtest_performance.csv` — Sharpe / drawdown for all strategy variants


In [ ]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn xgboost yfinance scipy


In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.6f}'.format)
np.random.seed(42)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titleweight'] = 'bold'

# ── Config ─────────────────────────────────────────────────────────
FAST_MODE      = True
FAST_STOCKS    = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA']
VIX_HIGH_THRESH = 25.0   # above this VIX, shift weight toward T+15
VIX_MED_THRESH  = 18.0   # below this VIX, shift weight toward T+1

print('[OK] Config loaded')


In [ ]:
# ── Path resolution ──────────────────────────────────────────────
cwd = Path.cwd()
if cwd.name == 'implementation' and cwd.parent.name == '02_stock_price_regression':
    reg_root = cwd.parent
elif cwd.name == '02_stock_price_regression':
    reg_root = cwd
elif (cwd / 'project_folder' / '02_stock_price_regression').exists():
    reg_root = cwd / 'project_folder' / '02_stock_price_regression'
else:
    reg_root = cwd

workspace_root = reg_root.parent
data_dir  = reg_root / 'data'
graph_dir = reg_root / 'graph'
data_dir.mkdir(parents=True, exist_ok=True)
graph_dir.mkdir(parents=True, exist_ok=True)
print(f'Data dir : {data_dir}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Load T+1 signal from 02D
# ══════════════════════════════════════════════════════════════════
# 02D stores per-stock evaluation in 02D_stock_level_eval.csv.
# For the per-day signal we need to reconstruct it from the test data.
# We use the 02D cluster-news features as a proxy dataset.

# Primary path: 02D saved evaluation file
d02d_eval_path    = data_dir / '02D_stock_level_eval.csv'
d02d_news_path    = data_dir / '02D_cluster_news_features.csv'
d02d_xgb_path     = data_dir / '02D_xgb_baseline_vs_clusternews.csv'

# Secondary paths for raw price data
cluster_data_dir  = workspace_root / '03_stock_clustering_analysis' / 'data'
price_candidates  = [
    cluster_data_dir / 'sp500_raw.csv',
    data_dir / 'sp500_raw.csv',
    data_dir / '02C_sp500_raw.csv',
]
price_path = next((p for p in price_candidates if p.exists()), None)

if price_path is None:
    raise FileNotFoundError(
        'Cannot find sp500_raw.csv. Please run 02C or 02D first.')

raw = pd.read_csv(price_path)
raw.columns = [c.strip() for c in raw.columns]
if 'name' in raw.columns and 'Name' not in raw.columns:
    raw.rename(columns={'name': 'Name'}, inplace=True)
close_col = next((c for c in raw.columns if c.lower() == 'close'), 'close')
if close_col != 'Close':
    raw.rename(columns={close_col: 'Close'}, inplace=True)
raw['date'] = pd.to_datetime(raw['date'], errors='coerce').dt.normalize()
raw = raw.dropna(subset=['Name', 'date']).sort_values(['Name', 'date']).reset_index(drop=True)

if FAST_MODE:
    raw = raw[raw['Name'].isin(FAST_STOCKS)].copy()

# Compute daily log return (actual T+1 realized return)
raw['ret_1d'] = raw.groupby('Name')['Close'].pct_change(1)
raw['log_ret_1d'] = np.log(raw['Close'] / raw.groupby('Name')['Close'].shift(1))

# ── Load 02D sentiment features and create a simple T+1 proxy signal ─
if d02d_news_path.exists():
    news_feat = pd.read_csv(d02d_news_path)
    news_feat['date'] = pd.to_datetime(news_feat['date'], errors='coerce').dt.normalize()
    news_feat['Name'] = news_feat['Name'].astype(str).str.strip()
    # Primary sentiment proxy from 02D: cluster_sentiment_10d or direct_sentiment_10d
    sent_col = next(
        (c for c in ['cluster_sentiment_10d', 'direct_sentiment_10d', 'macro_sentiment_10d']
         if c in news_feat.columns), None
    )
    if sent_col:
        news_feat = news_feat[['Name', 'date', sent_col]].rename(
            columns={sent_col: 'sent_02d'})
        raw = raw.merge(news_feat, on=['Name', 'date'], how='left')
        raw['sent_02d'] = raw['sent_02d'].fillna(0.0)
    else:
        raw['sent_02d'] = 0.0
else:
    raw['sent_02d'] = 0.0

# ── Load 02E multi-horizon predictions ───────────────────────────
e_pred_path = data_dir / '02E_multi_horizon_predictions.csv'
e_feat_path = data_dir / '02E_trend_features.csv'

if e_pred_path.exists():
    e_preds = pd.read_csv(e_pred_path)
    e_preds['date'] = pd.to_datetime(e_preds['date'], errors='coerce').dt.normalize()
    e_preds['Name'] = e_preds['Name'].astype(str).str.strip()
    print(f'[OK] 02E predictions loaded: {len(e_preds):,} rows')
    print('Columns:', e_preds.columns.tolist())
else:
    print('[WARN] 02E predictions not found — creating synthetic signal for demonstration')
    # Synthetic: simple lagged return as a stand-in for 02E T+15 prediction
    e_preds = raw[['Name', 'date']].copy()
    e_preds['pred_T5']  = raw.groupby('Name')['log_ret_1d'].transform(
        lambda x: x.rolling(5, min_periods=2).mean().shift(1)).values
    e_preds['pred_T10'] = raw.groupby('Name')['log_ret_1d'].transform(
        lambda x: x.rolling(10, min_periods=3).mean().shift(1)).values
    e_preds['pred_T15'] = raw.groupby('Name')['log_ret_1d'].transform(
        lambda x: x.rolling(15, min_periods=5).mean().shift(1)).values
    e_preds['true_T5']  = np.log(
        raw.groupby('Name')['Close'].transform(lambda x: x.shift(-5)) / raw['Close'])
    e_preds['true_T10'] = np.log(
        raw.groupby('Name')['Close'].transform(lambda x: x.shift(-10)) / raw['Close'])
    e_preds['true_T15'] = np.log(
        raw.groupby('Name')['Close'].transform(lambda x: x.shift(-15)) / raw['Close'])

print(f'Price rows: {len(raw):,}')
print(raw[['Name','date','Close','ret_1d','sent_02d']].head(3))


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Build merged signal DataFrame
# ══════════════════════════════════════════════════════════════════

# We need on each row:
#   date, Name,
#   t1_signal   (sign of predicted T+1 return: +1 or −1)
#   t15_pred    (raw 02E T+15 log-return prediction)
#   sent_vel    (sentiment velocity from 02E if available)
#   actual_ret  (realized T+1 log return — used for backtest PnL)

# T+1 signal: use the 02D stacked prediction if available; else use lagged momentum
# Proxy for T+1 prediction: 5-day EMA of daily returns (simple carry signal)
raw['t1_proxy'] = raw.groupby('Name')['log_ret_1d'].transform(
    lambda x: x.ewm(span=5, adjust=False).mean().shift(1)
)

# Merge 02E T+15 prediction
sig_df = raw[['Name', 'date', 'log_ret_1d', 't1_proxy', 'sent_02d']].copy()
sig_df = sig_df.merge(
    e_preds[['Name', 'date', 'pred_T15', 'true_T15']].copy(),
    on=['Name', 'date'], how='inner'
)

# ── Load sentiment velocity from 02E trend features if available ─
if e_feat_path.exists():
    e_feat = pd.read_csv(e_feat_path)
    e_feat['date'] = pd.to_datetime(e_feat['date'], errors='coerce').dt.normalize()
    if 'sentiment_velocity_10d' in e_feat.columns:
        sig_df = sig_df.merge(
            e_feat[['Name', 'date', 'sentiment_velocity_10d']],
            on=['Name', 'date'], how='left'
        )
        sig_df['sentiment_velocity_10d'] = sig_df['sentiment_velocity_10d'].fillna(0.0)
    else:
        sig_df['sentiment_velocity_10d'] = 0.0
else:
    sig_df['sentiment_velocity_10d'] = 0.0

sig_df = sig_df.dropna(subset=['log_ret_1d', 't1_proxy', 'pred_T15'])
sig_df = sig_df.sort_values(['Name', 'date']).reset_index(drop=True)

print(f'Signal DataFrame: {len(sig_df):,} rows')
print(sig_df.head(3).to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# VIX-Conditional Weighting
# ══════════════════════════════════════════════════════════════════
# Attempt to fetch real VIX data via yfinance; fall back to implied
# volatility proxy (cross-sectional average realized vol).

import yfinance as yf

try:
    vix_raw = yf.download('^VIX',
                          start=sig_df['date'].min().strftime('%Y-%m-%d'),
                          end=(sig_df['date'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d'),
                          progress=False)
    vix_raw = vix_raw[['Close']].rename(columns={'Close': 'VIX'}).reset_index()
    vix_raw.columns = ['date', 'VIX']
    vix_raw['date'] = pd.to_datetime(vix_raw['date']).dt.normalize()
    # Handle MultiIndex columns from yfinance
    if isinstance(vix_raw.columns, pd.MultiIndex):
        vix_raw.columns = vix_raw.columns.get_level_values(0)
    vix_raw = vix_raw.dropna()
    print(f'[OK] VIX data fetched: {len(vix_raw):,} rows')
    vix_available = len(vix_raw) > 0
except Exception as e:
    print(f'[WARN] VIX fetch failed ({e}) — using realized vol proxy')
    vix_available = False

if not vix_available:
    # Proxy: 20-day rolling cross-sectional average of individual stock volatilities
    vix_proxy = (
        sig_df.groupby('date')['log_ret_1d']
        .std().reset_index()
        .rename(columns={'log_ret_1d': 'daily_xsec_vol'})
        .sort_values('date')
    )
    vix_proxy['VIX'] = (
        vix_proxy['daily_xsec_vol'].rolling(20, min_periods=5).mean() * np.sqrt(252) * 100
    )
    vix_raw = vix_proxy[['date', 'VIX']].dropna()
    print(f'[OK] VIX proxy computed: {len(vix_raw):,} rows')

# Merge VIX into signal DataFrame
sig_df = sig_df.merge(vix_raw, on='date', how='left')
sig_df['VIX'] = sig_df.groupby('Name')['VIX'].transform(
    lambda x: x.fillna(method='ffill').fillna(x.median()))

# VIX regime classification
sig_df['vix_regime'] = pd.cut(
    sig_df['VIX'],
    bins=[-np.inf, VIX_MED_THRESH, VIX_HIGH_THRESH, np.inf],
    labels=['Low', 'Medium', 'High']
)
print(sig_df['vix_regime'].value_counts().to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Meta-Signal Construction — Three Filters
# ══════════════════════════════════════════════════════════════════

def compute_meta_signal(df,
                        vix_high=VIX_HIGH_THRESH,
                        vix_med=VIX_MED_THRESH):
    """
    Compute the 02F meta-signal combining T+1 (02D proxy) and T+15 (02E).

    Returns a copy of df with new columns:
        w_t1           : weight on T+1 signal  (0-1)
        w_t15          : weight on T+15 signal (0-1)
        regime_filter  : 0 if T+15 opposes T+1 direction strongly
        vel_align      : 0 if sentiment velocity conflicts with T+1 direction
        meta_score     : raw combined score (before sign)
        meta_signal    : final position {+1, -1, 0}
    """
    df = df.copy()

    # ── 1) VIX-Conditional weights ────────────────────────────────
    # High VIX → trust T+15; Low VIX → trust T+1
    w_t1 = np.where(
        df['VIX'] > vix_high, 0.30,
        np.where(df['VIX'] < vix_med, 0.70, 0.50)
    )
    w_t15 = 1.0 - w_t1
    df['w_t1']  = w_t1
    df['w_t15'] = w_t15

    # ── 2) Regime filter ─────────────────────────────────────────
    # If T+15 is negative but T+1 is positive (or vice versa),
    # discount based on |T+15_pred| × realized volatility
    t1_dir  = np.sign(df['t1_proxy'])
    t15_dir = np.sign(df['pred_T15'])
    aligned = (t1_dir == t15_dir).astype(float)  # 1 = agree, 0 = disagree

    # Rolling realized vol per stock
    df['real_vol'] = df.groupby('Name')['log_ret_1d'].transform(
        lambda x: x.rolling(20, min_periods=5).std().fillna(x.std()))

    # discount factor = 1 - |T+15_pred| × vol × (1 - aligned)
    # → discounts signal when T+1 and T+15 disagree AND T+15 is strong
    df['regime_filter'] = np.clip(
        1.0 - (df['pred_T15'].abs() * df['real_vol'] * 10 * (1 - aligned)), 0.0, 1.0)

    # ── 3) Sentiment velocity alignment ──────────────────────────
    # Discount T+1 if sentiment velocity contradicts T+1 direction
    vel_sign = np.sign(df['sentiment_velocity_10d'].replace(0, np.nan).fillna(0))
    vel_align = np.where(
        vel_sign == 0, 1.0,             # no velocity data → neutral
        np.where(vel_sign == t1_dir, 1.0, 0.75)  # conflict → 25% discount
    )
    df['vel_align'] = vel_align

    # ── Combined meta-score ───────────────────────────────────────
    # Weighted sum of T+1 and T+15 signals, filtered by regime and velocity
    t1_contrib  = df['w_t1']  * df['t1_proxy']
    t15_contrib = df['w_t15'] * df['pred_T15']
    df['meta_score'] = (
        (t1_contrib + t15_contrib)
        * df['regime_filter']
        * df['vel_align']
    )

    # ── Final position ─────────────────────────────────────────────
    df['meta_signal'] = np.sign(df['meta_score'])

    return df


sig_df = compute_meta_signal(sig_df)

# Also compute pure T+1 and T+15 signals as baselines
sig_df['signal_t1']  = np.sign(sig_df['t1_proxy'])
sig_df['signal_t15'] = np.sign(sig_df['pred_T15'])

print('Meta-signal computed.')
print(sig_df[['Name', 'date', 'VIX', 'w_t1', 'w_t15', 'regime_filter',
              'vel_align', 'meta_score', 'meta_signal']].head(10).to_string())


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Optional: Trained Meta-Learner (Ridge Regression)
# Learns optimal weights from data instead of fixed VIX thresholds
# ══════════════════════════════════════════════════════════════════

# Build meta-features for the learned meta-learner
meta_feat_cols = [
    't1_proxy',          # T+1 signal
    'pred_T15',          # T+15 signal
    'VIX',               # market stress level
    'sentiment_velocity_10d',  # narrative momentum
    'real_vol',          # realized vol (already computed)
]
sig_df['t1_t15_disagree'] = np.abs(
    np.sign(sig_df['t1_proxy']) - np.sign(sig_df['pred_T15']))  # 0 or 2
# Signal disagreement under stress
sig_df['stress_disagree'] = sig_df['VIX'] * sig_df['t1_t15_disagree']
meta_feat_cols += ['t1_t15_disagree', 'stress_disagree']

# Temporal split for meta-learner (same 80/20 logic)
all_dates = sig_df['date'].sort_values().unique()
split_date = pd.Timestamp(np.quantile(
    all_dates.astype(np.int64), 0.80, interpolation='nearest'))

meta_train = sig_df[sig_df['date'] <= split_date].dropna(subset=meta_feat_cols + ['log_ret_1d']).copy()
meta_test  = sig_df[sig_df['date'] >  split_date].dropna(subset=meta_feat_cols + ['log_ret_1d']).copy()

print(f'Meta-learner train: {len(meta_train):,}  test: {len(meta_test):,}')

if len(meta_train) > 30 and len(meta_test) > 10:
    X_meta_train = meta_train[meta_feat_cols].fillna(0).replace([np.inf, -np.inf], 0)
    X_meta_test  = meta_test[meta_feat_cols].fillna(0).replace([np.inf, -np.inf], 0)

    scaler_m = StandardScaler()
    X_mt = scaler_m.fit_transform(X_meta_train)
    X_me = scaler_m.transform(X_meta_test)

    # Target: realized T+1 return (what we want to maximize the sign of)
    y_meta_train = meta_train['log_ret_1d'].values
    y_meta_test  = meta_test['log_ret_1d'].values

    # Ridge regression meta-learner
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(X_mt, y_meta_train)
    meta_preds_score = meta_model.predict(X_me)
    meta_test = meta_test.copy()
    meta_test['learned_signal'] = np.sign(meta_preds_score)

    # XGBoost meta-learner (stronger)
    xgb_meta = XGBRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
        random_state=42, verbosity=0)
    xgb_meta.fit(X_mt, y_meta_train)
    xgb_meta_preds = xgb_meta.predict(X_me)
    meta_test['xgb_learned_signal'] = np.sign(xgb_meta_preds)

    # Merge back into sig_df
    meta_signal_map = meta_test.set_index(['Name', 'date'])[['learned_signal', 'xgb_learned_signal']]
    sig_df = sig_df.join(meta_signal_map, on=['Name', 'date'], how='left')
    sig_df['learned_signal']     = sig_df['learned_signal'].fillna(sig_df['meta_signal'])
    sig_df['xgb_learned_signal'] = sig_df['xgb_learned_signal'].fillna(sig_df['meta_signal'])

    print('Meta-learner coefficients (Ridge):')
    for feat, coef in sorted(zip(meta_feat_cols, meta_model.coef_),
                             key=lambda x: abs(x[1]), reverse=True):
        print(f'  {feat:35s}: {coef:+.5f}')
    meta_learner_available = True
else:
    print('[WARN] Insufficient data for meta-learner — using rule-based signal only')
    sig_df['learned_signal']     = sig_df['meta_signal']
    sig_df['xgb_learned_signal'] = sig_df['meta_signal']
    meta_learner_available = False


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Backtest Engine
# ══════════════════════════════════════════════════════════════════

def run_backtest(signal_series, realized_returns, dates, name='Strategy'):
    """
    Long when signal=+1, short when signal=−1, flat when signal=0.
    Returns a dict of performance metrics.
    """
    s = np.asarray(signal_series, dtype=float)
    r = np.asarray(realized_returns, dtype=float)
    d = np.asarray(dates)

    pnl = s * r
    equity = (1 + pnl).cumprod()
    bh_equity = (1 + r).cumprod()

    sharpe = (pnl.mean() / (pnl.std() + 1e-8)) * np.sqrt(252) if pnl.std() > 0 else 0.0

    running_max = np.maximum.accumulate(equity)
    dd = equity / running_max - 1.0
    max_dd = float(dd.min())

    bh_sharpe = (r.mean() / (r.std() + 1e-8)) * np.sqrt(252) if r.std() > 0 else 0.0
    bh_running_max = np.maximum.accumulate(bh_equity)
    bh_dd = bh_equity / bh_running_max - 1.0
    bh_max_dd = float(bh_dd.min())

    dhr = float(np.mean(np.sign(r[s != 0]) == s[s != 0])) if (s != 0).sum() > 0 else 0.5

    return {
        'name': name,
        'sharpe': round(sharpe, 4),
        'max_drawdown': round(max_dd, 4),
        'dhr': round(dhr, 4),
        'total_return': round(float(equity[-1] - 1), 4),
        'bh_sharpe': round(bh_sharpe, 4),
        'bh_max_drawdown': round(bh_max_dd, 4),
        'bh_return': round(float(bh_equity[-1] - 1), 4),
        'n_trades': int((s != 0).sum()),
        'equity': equity,
        'bh_equity': bh_equity,
        'drawdown': dd,
        'bh_drawdown': bh_dd,
        'dates': d,
        'pnl': pnl,
    }


# Only backtest on the test period (post split_date)
test_sig = sig_df[sig_df['date'] > split_date].copy()

# For portfolio backtest: aggregate daily across all stocks (equal weight)
port_daily = test_sig.sort_values(['date', 'Name'])
port_agg = port_daily.groupby('date').agg(
    signal_t1=('signal_t1', 'mean'),
    signal_t15=('signal_t15', 'mean'),
    meta_signal=('meta_signal', 'mean'),
    learned_signal=('learned_signal', 'mean'),
    xgb_signal=('xgb_learned_signal', 'mean'),
    actual_ret=('log_ret_1d', 'mean'),
).sort_values('date').reset_index()

# Binarise aggregated signals
for col in ['signal_t1', 'signal_t15', 'meta_signal', 'learned_signal', 'xgb_signal']:
    port_agg[f'{col}_bin'] = np.sign(port_agg[col])

print(f'Portfolio backtest rows: {len(port_agg):,}  ({port_agg["date"].min().date()} → {port_agg["date"].max().date()})')


In [ ]:
# ── Run all strategy backtests ───────────────────────────────────
bt_results = {}

strategies = {
    'T+1 Only (02D proxy)':     port_agg['signal_t1_bin'],
    'T+15 Only (02E)':          port_agg['signal_t15_bin'],
    'Rule-Based Meta (02F)':    port_agg['meta_signal_bin'],
    'Learned Meta Ridge (02F)': port_agg['learned_signal_bin'],
    'Learned Meta XGB (02F)':   port_agg['xgb_signal_bin'],
}

for name, signal_col in strategies.items():
    bt = run_backtest(
        signal_col.values,
        port_agg['actual_ret'].values,
        port_agg['date'].values,
        name=name
    )
    bt_results[name] = bt
    print(f'{name:35s}  Sharpe={bt["sharpe"]:+.3f}  MaxDD={bt["max_drawdown"]:+.2%}  '
          f'Return={bt["total_return"]:+.2%}  DHR={bt["dhr"]:.1%}  Trades={bt["n_trades"]}')

# Print Buy & Hold for reference
bh = bt_results['T+1 Only (02D proxy)']
print(f'{"Buy & Hold":35s}  Sharpe={bh["bh_sharpe"]:+.3f}  MaxDD={bh["bh_max_drawdown"]:+.2%}  '
      f'Return={bh["bh_return"]:+.2%}')


In [ ]:
# ── Cumulative Return Chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 6))
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
for (name, bt), color in zip(bt_results.items(), colors):
    ax.plot(bt['dates'], bt['equity'], label=name, linewidth=1.8, color=color)

# Buy & hold
first_bt = list(bt_results.values())[0]
ax.plot(first_bt['dates'], first_bt['bh_equity'],
        label='Buy & Hold', linewidth=2.0, color='black', linestyle='--')

ax.set_title('02F: Cumulative Returns — Strategy Comparison (Equal-Weight Portfolio)')
ax.set_xlabel('Date')
ax.set_ylabel('Equity (Starting = 1.0)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(graph_dir / '02F_01_cumulative_returns.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Drawdown Chart ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 5))
for (name, bt), color in zip(list(bt_results.items())[:4], colors[:4]):
    ax.plot(bt['dates'], bt['drawdown'] * 100,
            label=name, linewidth=1.5, color=color)
ax.fill_between(first_bt['dates'], first_bt['drawdown'] * 100, 0,
                color='black', alpha=0.08, label='Buy & Hold')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('02F: Maximum Drawdown Comparison')
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(graph_dir / '02F_02_drawdown.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Performance by VIX Regime ────────────────────────────────────
# Does the meta-signal improve most in high-VIX periods?
regime_records = []
for regime_label in ['Low', 'Medium', 'High']:
    mask = test_sig['vix_regime'] == regime_label
    if mask.sum() < 10:
        continue
    sub = test_sig[mask]
    sub_port = sub.groupby('date').agg(
        meta_signal=('meta_signal', lambda x: np.sign(x.mean())),
        signal_t1=('signal_t1', lambda x: np.sign(x.mean())),
        actual_ret=('log_ret_1d', 'mean'),
    ).reset_index()

    for sig_col, sig_name in [('meta_signal', '02F Meta'), ('signal_t1', 'T+1 Only')]:
        pnl = sub_port[sig_col] * sub_port['actual_ret']
        sharpe = (pnl.mean() / (pnl.std() + 1e-8)) * np.sqrt(252)
        dhr = (np.sign(sub_port['actual_ret']) == sub_port[sig_col]).mean()
        regime_records.append(dict(
            Regime=regime_label, Strategy=sig_name,
            Sharpe=round(float(sharpe), 3), DHR=round(float(dhr), 3),
            N_Days=len(sub_port)
        ))

regime_df = pd.DataFrame(regime_records)
print('Performance by VIX Regime:')
print(regime_df.to_string(index=False))

# Bar chart
if len(regime_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, metric in zip(axes, ['Sharpe', 'DHR']):
        pivot = regime_df.pivot(index='Regime', columns='Strategy', values=metric)
        pivot = pivot.reindex(['Low', 'Medium', 'High'])
        pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#3498db'])
        ax.set_title(f'{metric} by VIX Regime')
        ax.set_ylabel(metric)
        ax.set_xlabel('VIX Regime')
        ax.tick_params(axis='x', rotation=0)
        if metric == 'DHR':
            ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='Random')
        ax.legend(fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(graph_dir / '02F_03_vix_regime_performance.png', dpi=120, bbox_inches='tight')
    plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# False Breakout Suppression Analysis
# ══════════════════════════════════════════════════════════════════
# A false breakout: T+1 predicts UP (+1) but T+15 is negative (down move)

fb_test = test_sig.copy()
fb_test['is_false_breakout'] = (
    (fb_test['signal_t1'] > 0) & (fb_test['pred_T15'] < 0)
).astype(int)

fb_rate = fb_test['is_false_breakout'].mean()
print(f'False breakout rate in test set: {fb_rate:.1%}')

# Did the meta-signal correctly suppress these?
fb_days = fb_test[fb_test['is_false_breakout'] == 1]
meta_suppressed = (fb_days['meta_signal'] != fb_days['signal_t1']).mean()
print(f'Meta-signal suppression rate on false breakouts: {meta_suppressed:.1%}')

# P&L comparison: T+1 vs Meta on false-breakout days
if len(fb_days) > 0:
    pnl_t1_fb   = (fb_days['signal_t1']   * fb_days['log_ret_1d']).mean()
    pnl_meta_fb = (fb_days['meta_signal']  * fb_days['log_ret_1d']).mean()
    print(f'Avg daily PnL on false-breakout days:')
    print(f'  T+1 signal:  {pnl_t1_fb:+.5f}')
    print(f'  Meta signal: {pnl_meta_fb:+.5f}')
    print(f'  Improvement: {pnl_meta_fb - pnl_t1_fb:+.5f}')

# Bar chart: realized returns on false-breakout vs aligned days
fb_test['signal_type'] = np.where(
    fb_test['is_false_breakout'] == 1, 'False Breakout',
    np.where(fb_test['signal_t1'] == np.sign(fb_test['pred_T15']), 'Aligned', 'Neutral')
)
grouped = fb_test.groupby('signal_type')['log_ret_1d'].mean()
fig, ax = plt.subplots(figsize=(9, 5))
colors_fb = ['#e74c3c' if t == 'False Breakout' else '#2ecc71' for t in grouped.index]
ax.bar(grouped.index, grouped.values, color=colors_fb)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Average Realized Return by Signal Alignment Type')
ax.set_ylabel('Avg Log Return (T+1)')
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(graph_dir / '02F_04_false_breakout.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Per-Stock Backtest Results ───────────────────────────────────
stock_bt_records = []
for stock in sorted(test_sig['Name'].unique()):
    sub = test_sig[test_sig['Name'] == stock].sort_values('date')
    for sig_col, sig_name in [
        ('signal_t1',   'T+1'),
        ('signal_t15',  'T+15'),
        ('meta_signal', '02F Meta'),
    ]:
        pnl = sub[sig_col] * sub['log_ret_1d']
        sharpe = (pnl.mean() / (pnl.std() + 1e-8)) * np.sqrt(252)
        dhr = (np.sign(sub['log_ret_1d']) == sub[sig_col]).mean()
        stock_bt_records.append(dict(
            Stock=stock, Strategy=sig_name,
            Sharpe=round(float(sharpe), 4),
            DHR=round(float(dhr), 4),
            N=len(sub),
        ))

stock_bt_df = pd.DataFrame(stock_bt_records)
pivot_sharpe = stock_bt_df.pivot(index='Stock', columns='Strategy', values='Sharpe')
print('Per-Stock Sharpe Ratios:')
print(pivot_sharpe.to_string())

fig, ax = plt.subplots(figsize=(11, 5))
pivot_sharpe.plot(kind='bar', ax=ax, color=['#3498db', '#e74c3c', '#2ecc71'])
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Per-Stock Annualised Sharpe Ratio — T+1 vs T+15 vs 02F Meta')
ax.set_ylabel('Sharpe Ratio')
ax.tick_params(axis='x', rotation=0)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(graph_dir / '02F_05_per_stock_sharpe.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Save Output Artifacts
# ══════════════════════════════════════════════════════════════════

# 1) Full signal + prediction table
save_cols = ['Name', 'date', 'VIX', 'vix_regime',
             't1_proxy', 'pred_T15', 'sentiment_velocity_10d',
             'w_t1', 'w_t15', 'regime_filter', 'vel_align',
             'meta_score', 'meta_signal', 'signal_t1', 'signal_t15',
             'log_ret_1d']
sig_save = sig_df[save_cols].copy()
sig_save.to_csv(data_dir / '02F_meta_signal_predictions.csv', index=False)
print('[SAVED] 02F_meta_signal_predictions.csv')

# 2) Backtest performance summary
bt_summary_records = []
for name, bt in bt_results.items():
    bt_summary_records.append(dict(
        Strategy=name,
        Sharpe_Ratio=bt['sharpe'],
        Max_Drawdown=bt['max_drawdown'],
        Total_Return=bt['total_return'],
        DHR=bt['dhr'],
        N_Trades=bt['n_trades'],
        BH_Sharpe=bt['bh_sharpe'],
        BH_Return=bt['bh_return'],
    ))

bt_summary = pd.DataFrame(bt_summary_records)
bt_summary.to_csv(data_dir / '02F_backtest_performance.csv', index=False)
print('[SAVED] 02F_backtest_performance.csv')

# 3) Per-stock backtest
stock_bt_df.to_csv(data_dir / '02F_stock_level_backtest.csv', index=False)
print('[SAVED] 02F_stock_level_backtest.csv')

# ── Final summary ─────────────────────────────────────────────────
print('\n' + '='*70)
print('02F FINAL PERFORMANCE SUMMARY')
print('='*70)
print(bt_summary.to_string(index=False))
print('\nOutputs written:')
for f in ['02F_meta_signal_predictions.csv', '02F_backtest_performance.csv',
          '02F_stock_level_backtest.csv']:
    p = data_dir / f
    status = '✓' if p.exists() else '✗ NOT FOUND'
    print(f'  {status}  {f}')


In [ ]:
# ── Final Interpretation ─────────────────────────────────────────
print('\n' + '='*70)
print('SIGNAL QUALITY INTERPRETATION')
print('='*70)

best_strategy = bt_summary.sort_values('Sharpe_Ratio', ascending=False).iloc[0]
bh_sharpe = bt_summary['BH_Sharpe'].iloc[0]
bh_return = bt_summary['BH_Return'].iloc[0]

print(f'Best strategy:    {best_strategy["Strategy"]}')
print(f'Sharpe Ratio:     {best_strategy["Sharpe_Ratio"]:+.4f}  (Buy & Hold: {bh_sharpe:+.4f})')
print(f'Max Drawdown:     {best_strategy["Max_Drawdown"]:+.2%}')
print(f'Total Return:     {best_strategy["Total_Return"]:+.2%}  (Buy & Hold: {bh_return:+.2%})')
print(f'Directional Acc:  {best_strategy["DHR"]:.1%}')
print()
print('Regime filter effectiveness:')
print(f'  False breakout rate: {fb_rate:.1%}')
if len(fb_days) > 0:
    print(f'  Meta suppression:    {meta_suppressed:.1%} of false breakouts correctly discounted')
    print(f'  PnL improvement on false-breakout days: {pnl_meta_fb - pnl_t1_fb:+.6f}')


## Summary & Next Steps

### What 02F achieves
- **Regime-aware signal combination**: VIX-conditional weighting ensures the model trusts T+15 
  trend predictions during volatile periods (VIX > 25) and relies more on T+1 momentum in 
  calm markets (VIX < 18).
- **False breakout suppression**: The regime filter discounts bullish T+1 signals when T+15 
  predicts a medium-term reversal — reducing noise trades.
- **Sentiment velocity alignment**: When narrative momentum (from 02E) disagrees with the 
  short-term signal, the meta-signal applies a 25% discount.
- **Learned meta-learner**: Both Ridge and XGBoost meta-learners learn the optimal weighting 
  from historical data, capturing regime effects that fixed thresholds miss.

### Potential improvements
1. **Walk-forward meta-learner retraining** every 60 days to adapt to changing regimes
2. **Real VIX term structure** (VIX9D / VIX3M ratio) as a more nuanced fear gauge
3. **Options market features**: put/call ratio, implied vol skew — leading indicators of 
   institutional positioning that the model currently lacks
4. **Transaction cost simulation**: add bid-ask spread and market impact to PnL calculation
